# Laboratorio de Snowpark

Vas a trabajar con una tabla de 10 millones de ventas que creas tú mismo en Snowflake. En cada nivel primero escribes qué crees que va a pasar, luego corres la celda y al final explicas la diferencia. Las predicciones que fallan son las que más enseñan.

Corre las celdas en orden. Todo el cálculo pesado ocurre en el warehouse, no en este notebook.

## Preparación

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import FloatType, StringType, IntegerType, PandasSeriesType
import time

session = get_active_session()
session.use_schema("MI_BASE.MI_SCHEMA")   # cámbialo si tu schema es otro
print(session.get_current_role(), "|", session.get_current_warehouse(), "|", session.get_current_schema())

Esta celda crea la tabla `VENTAS_LAB` con 10 millones de filas. Tarda alrededor de un minuto en un warehouse X-Small.

In [ ]:
N = 10_000_000
session.sql(f"""
CREATE OR REPLACE TABLE VENTAS_LAB AS
SELECT SEQ8() + 1                                                     AS VENTA_ID,
       DATEADD(day, -UNIFORM(0, 729, RANDOM()), CURRENT_DATE())       AS FECHA,
       UNIFORM(1, 2000, RANDOM())                                     AS TIENDA_ID,
       ARRAY_CONSTRUCT('NORESTE','NOROESTE','CENTRO','OCCIDENTE','SUR')[UNIFORM(0,4,RANDOM())]::STRING AS REGION,
       ARRAY_CONSTRUCT('TRADICIONAL','MODERNO','B2B')[UNIFORM(0,2,RANDOM())]::STRING             AS CANAL,
       'SKU-' || LPAD(UNIFORM(1, 400, RANDOM())::STRING, 4, '0')       AS SKU,
       UNIFORM(1, 48, RANDOM())                                       AS UNIDADES,
       ROUND(UNIFORM(8::FLOAT, 120::FLOAT, RANDOM()), 2)              AS PRECIO
FROM TABLE(GENERATOR(ROWCOUNT => {N}))
""").collect()

ventas = session.table("VENTAS_LAB")
print(f"{ventas.count():,} filas")
ventas.show(5)

## Nivel 1 · El plan

Vamos a definir una transformación y a mirar qué hizo Snowpark con ella.

**Predice antes de correr.** Cuando corras la celda de abajo, ¿cuántas consultas se ejecutan en Snowflake? ¿Cuántas filas viajan a este notebook?

_Tu predicción:_ 

In [ ]:
res = (ventas
       .filter(F.col("REGION") == "NORESTE")
       .with_column("IMPORTE", F.col("UNIDADES") * F.col("PRECIO"))
       .group_by("CANAL")
       .agg(F.sum("IMPORTE").alias("IMPORTE_TOTAL"),
            F.count("*").alias("VENTAS")))

print(res.queries["queries"][-1])   # el SQL que Snowpark generó

In [ ]:
res.explain()   # el plan que usará Snowflake

**Explica.** ¿Dónde aparece tu `filter` en el SQL? ¿Y el `with_column`?

_Tu explicación:_ 

## Nivel 2 · Ejecución diferida, medida

`session.query_history()` registra cada consulta que el notebook manda a Snowflake. Lo usamos para contar.

**Predice antes de correr.** Al definir `res2` en el primer bloque, ¿cuántas consultas se registran? ¿Y al llamar `.show()`?

_Tu predicción:_ 

In [ ]:
with session.query_history() as historial:
    res2 = ventas.filter(F.col("CANAL") == "B2B").select("TIENDA_ID", "UNIDADES")
    res2 = res2.group_by("TIENDA_ID").agg(F.sum("UNIDADES").alias("UNIDADES"))
print("consultas al definir:", len(historial.queries))

with session.query_history() as historial:
    res2.sort(F.col("UNIDADES").desc()).show(5)
print("consultas al mostrar:", len(historial.queries))
for q in historial.queries:
    print("  ", q.sql_text[:160].replace("\n", " "))

**Predice antes de correr.** Ahora llamamos `.count()` y luego `.show()` sobre el mismo DataFrame. ¿Cuántas veces se recalcula la agregación sobre los 10 millones de filas?

_Tu predicción:_ 

In [ ]:
with session.query_history() as h:
    res2.count()
    res2.show(3)
print("sin cache:", len(h.queries), "consultas, cada una recorre VENTAS_LAB")

guardado = res2.cache_result()      # materializa el resultado en una tabla temporal
with session.query_history() as h:
    guardado.count()
    guardado.show(3)
print("con cache_result:", len(h.queries), "consultas, ahora sobre 2,000 filas")

In [ ]:
# Trampa frecuente: show() imprime pero devuelve None
x = res2.show(1)
print(type(x))

**Explica.** ¿Por qué Snowpark espera hasta la acción en lugar de ejecutar cada línea?

_Tu explicación:_ 

## Nivel 3 · Llevar el cálculo al dato o traer el dato

**Predice antes de correr.** Vamos a sumar el importe por región de dos formas: en Snowflake sobre 10 millones de filas, y con pandas después de traer 2 millones con `to_pandas()`. ¿Cuál tarda más y cuánta memoria ocupa la segunda?

_Tu predicción:_ 

In [ ]:
t0 = time.perf_counter()
en_snowflake = (ventas
                .group_by("REGION")
                .agg(F.sum(F.col("UNIDADES") * F.col("PRECIO")).alias("IMPORTE"))
                .collect())
t_snow = time.perf_counter() - t0

t0 = time.perf_counter()
pdf = ventas.limit(2_000_000).to_pandas()
pdf["IMPORTE"] = pdf["UNIDADES"] * pdf["PRECIO"]
en_pandas = pdf.groupby("REGION")["IMPORTE"].sum()
t_pandas = time.perf_counter() - t0

print(f"Snowflake, 10 M filas: {t_snow:5.1f} s · filas recibidas: {len(en_snowflake)}")
print(f"pandas,     2 M filas: {t_pandas:5.1f} s · memoria: {pdf.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
del pdf

**Explica.** Si la tabla tuviera 500 millones de filas, ¿qué le pasaría a la versión con pandas?

_Tu explicación:_ 

## Nivel 4 · Función nativa, UDF y UDF vectorizada

Clasificamos cada venta por precio (BAJO, MEDIO, ALTO) de tres formas, sobre una muestra de 1 millón de filas. Si tu rol no puede crear funciones, la celda te lo dice y puedes seguir con el nivel 5.

**Predice antes de correr.** Ordena las tres de la más rápida a la más lenta.

_Tu predicción:_ 

In [ ]:
muestra = ventas.limit(1_000_000).cache_result()

def cronometra(df, etiqueta):
    t0 = time.perf_counter()
    filas = df.group_by("CAT").count().collect()
    print(f"{etiqueta:<16} {time.perf_counter() - t0:5.1f} s  {sorted((r['CAT'], r['COUNT']) for r in filas)}")

# 1. Función nativa: se traduce a un CASE en SQL
nativa = muestra.with_column("CAT", F.when(F.col("PRECIO") >= 80, "ALTO")
                                     .when(F.col("PRECIO") >= 30, "MEDIO")
                                     .otherwise("BAJO"))
cronometra(nativa, "Nativa")

# 2. UDF de Python: una llamada por fila
def categoria(p: float) -> str:
    return "ALTO" if p >= 80 else ("MEDIO" if p >= 30 else "BAJO")

# 3. UDF vectorizada: recibe lotes como pandas.Series
import pandas as pd
def categoria_vec(p: pd.Series) -> pd.Series:
    return pd.cut(p, bins=[-float("inf"), 30, 80, float("inf")],
                  labels=["BAJO", "MEDIO", "ALTO"], right=False).astype(str)

try:
    cat_udf = session.udf.register(categoria, return_type=StringType(), input_types=[FloatType()],
                                   name="CATEGORIA_LAB", is_permanent=False, replace=True)
    cronometra(muestra.with_column("CAT", cat_udf(F.col("PRECIO").cast("float"))), "UDF")

    cat_vec = session.udf.register(categoria_vec, return_type=PandasSeriesType(StringType()),
                                   input_types=[PandasSeriesType(FloatType())],
                                   name="CATEGORIA_VEC_LAB", is_permanent=False, replace=True,
                                   packages=["pandas"])
    cronometra(muestra.with_column("CAT", cat_vec(F.col("PRECIO").cast("float"))), "UDF vectorizada")
except Exception as e:
    print("Tu rol no pudo crear la UDF. Mensaje de Snowflake:\n", str(e)[:300])

**Explica.** ¿Por qué la nativa gana aunque la UDF haga exactamente lo mismo?

_Tu explicación:_ 

## Nivel 5 · Un pipeline que se puede volver a correr

**Predice antes de correr.** La función de abajo construye un resumen mensual por región. ¿Cuántas filas tendrá la tabla después de correrla dos veces con `overwrite`? ¿Y si luego la corres con `append`?

_Tu predicción:_ 

In [ ]:
def construir_resumen(session, modo: str = "overwrite") -> int:
    v = session.table("VENTAS_LAB")
    resumen = (v.with_column("MES", F.date_trunc("month", F.col("FECHA")))
                .with_column("IMPORTE", F.col("UNIDADES") * F.col("PRECIO"))
                .group_by("MES", "REGION")
                .agg(F.sum("IMPORTE").alias("IMPORTE"), F.count("*").alias("VENTAS")))
    resumen.write.mode(modo).save_as_table("RESUMEN_MENSUAL_LAB")
    return session.table("RESUMEN_MENSUAL_LAB").count()

print("overwrite 1:", construir_resumen(session))
print("overwrite 2:", construir_resumen(session))
print("append     :", construir_resumen(session, "append"))

El mismo pipeline, empaquetado como stored procedure para que corra completo dentro de Snowflake. Si tu rol no puede crearlo, la celda te lo dice.

In [ ]:
try:
    session.sproc.register(construir_resumen, name="CONSTRUIR_RESUMEN_LAB",
                           return_type=IntegerType(), input_types=[StringType()],
                           is_permanent=False, replace=True,
                           packages=["snowflake-snowpark-python"])
    print("desde el procedure:", session.call("CONSTRUIR_RESUMEN_LAB", "overwrite"))
except Exception as e:
    print("Tu rol no pudo crear el procedure. Mensaje de Snowflake:\n", str(e)[:300])

**Explica.** ¿Qué modo usarías en un job que corre cada noche y por qué? ¿Qué cambiaría si en vez de recalcular todo quisieras agregar solo el día nuevo?

_Tu explicación:_ 

## Desafío · Migrar un job de PySpark

Este es un job real de Databricks. Tradúcelo a Snowpark en la celda de abajo. La mayoría de los métodos tiene un equivalente directo, pero no siempre se llama igual.

```python
# PySpark
from pyspark.sql import functions as F
df = spark.read.table("ventas")
top = (df.where(F.col("canal") == "MODERNO")
         .withColumn("importe", F.col("unidades") * F.col("precio"))
         .groupBy("sku").agg(F.sum("importe").alias("importe"))
         .orderBy(F.desc("importe")).limit(10))
top.write.mode("overwrite").saveAsTable("top_sku")
```

In [ ]:
# Tu versión en Snowpark (usa VENTAS_LAB y guarda en TOP_SKU_LAB)
top = ...

## Bonus · pandas on Snowflake

Si vienes de pandas, puedes usar su sintaxis y que el cálculo corra en Snowflake. Necesitas agregar el paquete `modin` en **Packages** del notebook.

In [ ]:
try:
    import modin.pandas as mpd
    import snowflake.snowpark.modin.plugin   # conecta modin con Snowflake
    m = mpd.read_snowflake("VENTAS_LAB")
    print(m.groupby("REGION")["UNIDADES"].sum())
except ImportError:
    print("Agrega el paquete 'modin' en Packages del notebook y vuelve a correr la celda.")

## Limpieza

Corre esta celda al terminar para no dejar tablas de prueba en el schema.

In [ ]:
for t in ["VENTAS_LAB", "RESUMEN_MENSUAL_LAB", "TOP_SKU_LAB"]:
    session.sql(f"DROP TABLE IF EXISTS {t}").collect()
print("listo")